# Detection method comparison

Runs every changepoint detector back to back in one cell, on the same cached
runs, so the numbers are directly comparable.

**No MATLAB.** Reads committed caches only. Takes about two minutes.

**The protocol matters more than the ranking.** Each method's sensitivity
parameter is calibrated on the 20 null runs, where `pd` never changes so every
alarm is false. The chosen value is the most sensitive one that stays within
the false-alarm budget. Only then is latency measured, on runs that had no part
in calibration.

Latency cannot be compared without this. Any detector reaches zero latency by
firing immediately on every run; what stops it is the false-alarm budget, so
the budget has to be equal before the latencies mean anything.

The floor is **0.50 s**. The first window containing only post-switch data is
centred half a window after the switch, so no causal detector on this series
can beat that.

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Find the repo root from wherever the notebook was launched, so this works
# whether Jupyter started in the repo, in docs/, or anywhere above it.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        if str(_p / "src") not in sys.path:
            sys.path.insert(0, str(_p / "src"))
        break
else:
    raise SystemExit(
        "Could not find src/paths.py. Launch Jupyter from the repository "
        "root (the folder holding src/, cache/ and the notebooks)."
    )

import numpy as np
import detect_alt as da
import pd_switch as ps
import recheck_stim as rs

# ---------------------------------------------------------------- data
def load(name, keep=lambda s: True):
    r = rs.load_cached(name)
    if not r:
        raise SystemExit(f"no cache for {name} - check cache/ is populated")
    return rs.pair_average({s: v for s, v in r.items() if keep(s)})

NULLS = [load("runs_null_nostim"), load("runs_null_stim180_500")]
N_NULL = sum(len(s) for s in NULLS)

EVAL = {
    "unstimulated, held out": load("runs_switch", lambda s: s > 10),
    "180 Hz / 500 uA, up":    load("runs_dir_0.3to0.7_stim180_500"),
    "180 Hz / 500 uA, down":  load("runs_dir_0.7to0.3_stim180_500"),
}

# ------------------------------------------------------------- methods
# Each grid runs most-sensitive first, so calibration takes the first value
# that meets the budget.
METHODS = dict(da.METHODS)
METHODS["Max-split t (ours)"] = (ps.detect_cp2,    "thresh",  np.arange(1.0, 20.01, 0.25))
METHODS["k-means (ours)"]     = (ps.detect_kmeans, "min_gap", np.arange(0.01, 1.001, 0.01))

def fired(series, fn, **kw):
    """Alarms on a run with no switch: both 'ok' and 'false' are false alarms."""
    r = ps.evaluate(series, fn, **kw)
    return r["ok"] + len(r["false"])

def calibrate(fn, param, grid, budget):
    for x in grid:
        n = sum(fired(s, fn, **{param: float(x)}) for s in NULLS)
        if n <= budget:
            return float(x), n
    return None, None

# --------------------------------------------------- main run, budget 1
BUDGET = 1
print(f"CALIBRATION   {N_NULL} null runs, budget {BUDGET} false alarm(s)")
print("-" * 70)
def best_selectivity(fn, param, grid):
    """Lowest false-alarm count reachable while the detector can still fire.

    A method that meets the budget only by going silent has not been
    calibrated, it has been switched off. This finds the floor that matters:
    the fewest false alarms it achieves at any setting where it still detects
    real switches."""
    probe = EVAL["unstimulated, held out"]
    best = None
    for x in grid:
        if ps.evaluate(probe, fn, **{param: float(x)})["ok"] == 0:
            continue
        n = sum(fired(s, fn, **{param: float(x)}) for s in NULLS)
        best = n if best is None else min(best, n)
    return best

chosen, degenerate = {}, {}
for name, (fn, param, grid) in METHODS.items():
    x, n = calibrate(fn, param, grid, BUDGET)
    if x is None:
        degenerate[name] = "no setting meets the budget at all"
        print(f"  {name:<20} no setting meets the budget")
        continue
    if ps.evaluate(EVAL["unstimulated, held out"], fn, **{param: float(x)})["ok"] == 0:
        floor = best_selectivity(fn, param, grid)
        degenerate[name] = (f"meets the budget only by never firing; best it "
                            f"manages while still detecting is {floor}/{N_NULL}")
        print(f"  {name:<20} {param:<8} = {x:<8.3g}  SILENT at this setting - "
              f"best false-alarm rate while still firing is {floor}/{N_NULL}")
        continue
    chosen[name] = (fn, param, x)
    print(f"  {name:<20} {param:<8} = {x:<8.3g}  false alarms {n}/{N_NULL}")

if degenerate:
    print("\n  Excluded from the latency tables below, with the reason:")
    for k, v in degenerate.items():
        print(f"    {k}: {v}")
    print("  These are properties of the method on an 18-point series, not")
    print("  failures to detect. Both separate switches from noise too poorly")
    print("  to reach the budget without being turned off entirely.")

for label, series in EVAL.items():
    print(f"\n{label.upper()}   n = {len(series)}, switch at 5.0 s")
    print("-" * 70)
    print(f"  {'method':<20}{'caught':>9}{'latency':>10}{'sem':>8}{'early':>7}")
    rows = []
    for name, (fn, param, x) in chosen.items():
        r = ps.evaluate(series, fn, **{param: x})
        rows.append((r["latency"] if r["ok"] else np.inf, name, r))
    for _, name, r in sorted(rows):
        lat = f"{r['latency']:.2f}s" if r["ok"] else "--"
        sem = f"{r['sem']:.2f}" if r["ok"] > 1 and np.isfinite(r["sem"]) else "--"
        print(f"  {name:<20}{r['ok']:>6}/{r['n']}{lat:>10}{sem:>8}{len(r['false']):>7}")

# ------------------------------------------- robustness to the budget
# 20 null runs is a small calibration set, so a ranking that only holds at
# one budget is a ranking of the calibration noise.
print(f"\n\nSAME COMPARISON AT OTHER BUDGETS   caught / latency")
print("=" * 70)
for budget in (0, 1, 2):
    print(f"\nbudget {budget}/{N_NULL}")
    print(f"  {'method':<20}" + "".join(f"{k.split(',')[0][:16]:>20}" for k in EVAL))
    for name, (fn, param, grid) in METHODS.items():
        x, _ = calibrate(fn, param, grid, budget)
        if x is None:
            print(f"  {name:<20}{'(uncalibratable)':>20}")
            continue
        cells = []
        for s in EVAL.values():
            r = ps.evaluate(s, fn, **{param: x})
            cells.append(f"{r['ok']}/{r['n']} {r['latency']:.2f}s" if r["ok"]
                         else f"{r['ok']}/{r['n']}   --")
        print(f"  {name:<20}" + "".join(f"{c:>20}" for c in cells))

print("\n" + "=" * 70)
print("Latency is measured from the switch at 5.0 s; the floor is 0.50 s.")
print("'early' counts alarms before the switch, which are false alarms, not")
print("fast detections.")

In [ ]:
# ============================================================================
# EXTENDED RUN - every cached switch run, repeated calibration, paired
# bootstrap. Several minutes.
#
# The cell above uses 40 evaluation runs and calibrates once on 20 nulls.
# Three things could make its ranking an artifact rather than a result:
#   1. too few evaluation runs,
#   2. a threshold that happened to land well on those particular 20 nulls,
#   3. comparing means across runs when brain-to-brain variation is large.
# This cell addresses all three: 190 evaluation runs, 200 recalibrations on
# resampled nulls, and bootstrap confidence intervals on PAIRED per-brain
# latency differences.
#
# It works by computing each method's decision statistic once per run, then
# deriving the alarm time for any threshold by first crossing. Identical
# results to calling the detectors directly - asserted below - but it turns
# the bootstrap from hours into seconds.
# ============================================================================
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Find the repo root from wherever the notebook was launched, so this works
# whether Jupyter started in the repo, in docs/, or anywhere above it.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        if str(_p / "src") not in sys.path:
            sys.path.insert(0, str(_p / "src"))
        break
else:
    raise SystemExit(
        "Could not find src/paths.py. Launch Jupyter from the repository "
        "root (the folder holding src/, cache/ and the notebooks)."
    )

if "METHODS" not in globals():
    raise SystemExit(
        "Run the first cell first. This one reuses the detectors, the loaded "
        "runs and the calibration helpers it defines."
    )

import json
import numpy as np
import paths
from detect_alt import BURN, _baseline

rng = np.random.default_rng(0)

# ---------------------------------------------------- decision statistics
def tr_cusum(t, v, k=0.5):
    mu, sd = _baseline(v); s = np.full(len(v), -np.inf); sp = sn = 0.0
    for i in range(BURN, len(v)):
        z = (v[i] - mu) / sd
        sp = max(0.0, sp + z - k); sn = max(0.0, sn - z - k)
        s[i] = max(sp, sn)
    return s, "above"

def tr_ph(t, v, delta=0.05):
    mu, sd = _baseline(v); s = np.full(len(v), -np.inf)
    mp = mn = 0.0; lo_p = lo_n = 0.0
    for i in range(BURN, len(v)):
        z = (v[i] - mu) / sd
        mp += z - delta; mn += -z - delta
        lo_p = min(lo_p, mp); lo_n = min(lo_n, mn)
        s[i] = max(mp - lo_p, mn - lo_n)
    return s, "above"

def tr_ewma(t, v, lam=0.3):
    mu, sd = _baseline(v); s = np.full(len(v), -np.inf); z = mu
    for i in range(BURN, len(v)):
        z = lam * v[i] + (1 - lam) * z
        j = i - BURN + 1
        unit = sd * np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * j)))
        s[i] = abs(z - mu) / unit if unit > 0 else np.inf
    return s, "above"

def tr_shewhart(t, v):
    mu, sd = _baseline(v); s = np.full(len(v), -np.inf)
    for i in range(BURN, len(v)):
        s[i] = abs(v[i] - mu) / sd
    return s, "above"

def tr_bocpd(t, v, hazard=0.1, settle=2):
    mu0, sd = _baseline(v); s = np.full(len(v), -np.inf)
    R = np.array([1.0]); means = np.array([mu0]); counts = np.array([float(BURN)])
    for i in range(BURN, len(v)):
        x = v[i]
        pred = np.exp(-0.5 * ((x - means) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
        growth = R * pred * (1 - hazard); cp = float((R * pred * hazard).sum())
        newR = np.concatenate(([cp], growth)); tot = newR.sum()
        if tot <= 0: break
        R = newR / tot
        means = np.concatenate(([mu0], means + (x - means) / (counts + 1)))
        counts = np.concatenate(([1.0], counts + 1))
        if i >= BURN + settle: s[i] = R[0]
    return s, "above"

def tr_mw(t, v, w=5):
    from scipy.stats import mannwhitneyu
    s = np.full(len(v), np.inf)
    for i in range(2 * w - 1, len(v)):
        a = v[i - 2 * w + 1:i - w + 1]; b = v[i - w + 1:i + 1]
        try: s[i] = mannwhitneyu(a, b, alternative="two-sided").pvalue
        except ValueError: pass
    return s, "below"          # fires when p drops below alpha

def tr_adwin(t, v, min_size=3):
    s = np.full(len(v), -np.inf)
    for i in range(2 * min_size - 1, len(v)):
        h = v[:i + 1]; rngv = float(h.max() - h.min())
        if rngv <= 0: continue
        best = -np.inf
        for p in range(min_size, len(h) - min_size + 1):
            a, b = h[:p], h[p:]
            m = 1.0 / (1.0 / len(a) + 1.0 / len(b))
            best = max(best, abs(b.mean() - a.mean()) * np.sqrt(2.0 * m) / rngv)
        s[i] = best
    return s, "above"          # threshold is sqrt(log(2/delta))

def tr_cp2(t, v, min_size=3):
    s = np.full(len(v), -np.inf)
    for i in range(2 * min_size, len(v)):
        h = v[:i + 1]; best = 0.0
        for p in range(min_size, len(h) - min_size + 1):
            a, b = h[:p], h[p:]
            sd = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
            if sd > 0: best = max(best, abs(b.mean() - a.mean()) / sd)
        s[i] = best
    return s, "above"

def tr_kmeans(t, v, max_breaks=6, min_size=3):
    from sklearn.cluster import KMeans
    s = np.full(len(v), -np.inf)
    for i in range(2 * min_size, len(v)):
        hist = v[:i + 1].reshape(-1, 1)
        km = KMeans(2, n_init=10, random_state=0).fit(hist)
        lab, c = km.labels_, km.cluster_centers_.ravel()
        hi = lab == int(np.argmax(c))
        if min(hi.sum(), (~hi).sum()) < min_size: continue
        if (~hi[np.argmax(hi):]).sum() > max_breaks: continue
        s[i] = (c.max() - c.min()) / c.min()
    return s, "above"

TRAJ = {
    "CUSUM": (tr_cusum, lambda h: h), "Page-Hinkley": (tr_ph, lambda h: h),
    "EWMA": (tr_ewma, lambda L: L), "Shewhart": (tr_shewhart, lambda L: L),
    "BOCPD": (tr_bocpd, lambda x: x), "Mann-Whitney": (tr_mw, lambda a: a),
    "ADWIN": (tr_adwin, lambda d: np.sqrt(np.log(2.0 / d))),
    "Max-split t (ours)": (tr_cp2, lambda x: x),
    "k-means (ours)": (tr_kmeans, lambda g: g),
}
GRIDS = {k: METHODS[k][2] for k in TRAJ}

def alarm(times, stat, direction, thresh):
    idx = np.where(stat > thresh)[0] if direction == "above" else np.where(stat < thresh)[0]
    return times[idx[0]] + ps.HALF_WINDOW_S if len(idx) else np.nan

# ------------------------------------------------- self-check vs detectors
_probe = EVAL["unstimulated, held out"]
for nm, direct, param, val in [
    ("CUSUM", da.detect_cusum, "h", 9.25), ("EWMA", da.detect_ewma, "L", 4.2),
    ("Max-split t (ours)", ps.detect_cp2, "thresh", 6.0),
    ("k-means (ours)", ps.detect_kmeans, "min_gap", 0.2)]:
    f, conv = TRAJ[nm]
    for sd_ in list(_probe)[:5]:
        tt, vv = _probe[sd_]
        st, dr = f(tt, vv)
        a = alarm(tt, st, dr, conv(val)); b = direct(tt, vv, **{param: val})
        assert (np.isnan(a) and np.isnan(b)) or abs(a - b) < 1e-9, (nm, sd_, a, b)
print("self-check: trajectory alarms match the detectors exactly\n")

# -------------------------------------------------------------- load all
def switch_of(name):
    return json.loads(str(np.load(paths.cache(name) / "seed1.npz")["settings"]))["hold_ms"] / 1000.0

SKIP = {"runs_t3p", "runs_agents", "runs_selector", "runs_selector.bak", "runs_neural", "runs"}
ALL = []
for d in sorted(p.name for p in paths.cache_dirs("runs_*")):
    if d in SKIP or "null" in d: continue
    r = rs.load_cached(d)
    if not r: continue
    m = json.loads(str(np.load(paths.cache(d) / "seed1.npz")["settings"]))
    if m["pd_start"] == m["pd_end"]: continue
    sw = m["hold_ms"] / 1000.0
    for s, tv in rs.pair_average(r).items():
        ALL.append((f"{d}#{s}", tv, sw))
NULL_RUNS = [tv for s in NULLS for tv in s.values()]
print(f"{len(ALL)} switch runs across {len(set(k.split('#')[0] for k,_,_ in ALL))} conditions"
      f"   |   {len(NULL_RUNS)} null runs\n")

print("computing decision statistics", end="", flush=True)
STAT_SW, STAT_NU = {}, {}
for nm, (f, conv) in TRAJ.items():
    STAT_SW[nm] = [(k, tv[0], f(*tv)[0], f(*tv)[1], sw) for k, tv, sw in ALL]
    STAT_NU[nm] = [(tv[0], f(*tv)[0], f(*tv)[1]) for tv in NULL_RUNS]
    print(".", end="", flush=True)
print(" done\n")

def fp_count(nm, thresh):
    return sum(not np.isnan(alarm(t, s, d, thresh)) for t, s, d in STAT_NU[nm])

def calib(nm, budget, idx=None):
    conv = TRAJ[nm][1]
    rows = STAT_NU[nm] if idx is None else [STAT_NU[nm][i] for i in idx]
    for x in GRIDS[nm]:
        th = conv(float(x))
        if sum(not np.isnan(alarm(t, s, d, th)) for t, s, d in rows) <= budget:
            return float(x)
    return None

# ------------------------------- 1. how stable is the calibrated threshold
print("=" * 74)
print("1. THRESHOLD STABILITY   200 recalibrations on 10 nulls drawn at random")
print("=" * 74)
print(f"  {'method':<20}{'median':>10}{'10-90 pct':>18}{'full-set':>11}")
STAB = {}
for nm in TRAJ:
    picks = []
    for _ in range(200):
        idx = rng.choice(len(STAT_NU[nm]), size=10, replace=False)
        x = calib(nm, 0, idx)
        if x is not None: picks.append(x)
    full = calib(nm, 1)
    if not picks or full is None:
        print(f"  {nm:<20}{'uncalibratable':>39}"); continue
    p = np.percentile(picks, [10, 50, 90]); STAB[nm] = p
    print(f"  {nm:<20}{p[1]:>10.3g}{f'{p[0]:.3g} - {p[2]:.3g}':>18}{full:>11.3g}")

# ------------------- 2 and 3. all runs, at both false-alarm budgets
# Both budgets, because the comparison changes between them and reporting
# only one would be choosing the answer.
for BUD in (0, 1):
    print("\n" + "=" * 74)
    print(f"ALL {len(ALL)} SWITCH RUNS   budget {BUD}/{len(NULL_RUNS)} false alarms")
    print("=" * 74)
    print(f"  {'method':<20}{'caught':>10}{'latency':>10}{'sem':>8}{'early':>7}")
    LAT = {}
    for nm in TRAJ:
        x = calib(nm, BUD)
        if x is None:
            print(f"  {nm:<20}{'uncalibratable':>27}"); continue
        th = TRAJ[nm][1](float(x))
        lat, early = {}, 0
        for k, t, s, d, sw in STAT_SW[nm]:
            a = alarm(t, s, d, th)
            if np.isnan(a): continue
            if a < sw: early += 1
            else: lat[k] = a - sw
        LAT[nm] = lat
        if lat:
            v = np.array(list(lat.values()))
            print(f"  {nm:<20}{len(lat):>7}/{len(ALL)}{v.mean():>9.2f}s"
                  f"{v.std(ddof=1)/np.sqrt(len(v)):>8.2f}{early:>7}")
        else:
            print(f"  {nm:<20}{0:>7}/{len(ALL)}{'--':>10}{'--':>8}{early:>7}"
                  f"   (silent at this budget)")

    base = LAT.get("Max-split t (ours)", {})
    print(f"\n  paired against the max-split t-test, same brain, both caught it")
    print(f"  negative = alarms earlier than ours; 95% bootstrap CI, 5000 resamples\n")
    print(f"  {'method':<20}{'paired n':>9}{'mean diff':>12}{'95% CI':>20}{'earlier':>10}")
    for nm in TRAJ:
        if nm == "Max-split t (ours)" or not LAT.get(nm): continue
        keys = sorted(set(base) & set(LAT[nm]))
        if len(keys) < 10:
            print(f"  {nm:<20}{len(keys):>9}   too few paired runs"); continue
        d = np.array([LAT[nm][k] - base[k] for k in keys])
        bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(5000)])
        lo, hi = np.percentile(bs, [2.5, 97.5])
        print(f"  {nm:<20}{len(keys):>9}{d.mean():>11.2f}s"
              f"{f'[{lo:+.2f}, {hi:+.2f}]':>20}{(d < 0).mean():>9.0%}")

print("\n" + "=" * 74)
print("Read the interval, not the mean. An interval straddling zero means the")
print("two methods are not separated by this data, however different the means")
print("look. 'earlier' is the fraction of individual brains on which the method")
print("alarmed before ours, which is what a paired comparison actually tests.")
print("\nThe two budgets answer different questions. At 1 in 20 the comparison")
print("is about speed. At 0 in 20 it is about how much speed a method keeps")
print("when it is not allowed any false alarms at all.")

In [ ]:
# ============================================================================
# SHOULD THE DETECTOR BECOME CUSUM?
#
# Every row of docs/DETECTION_RESULTS.md recomputed under three detectors:
# the one-sided max-split t-test used for that document's tables, the
# two-sided version section 5 recommends, and CUSUM. All calibrated the same
# way, on the 20 null runs within one false alarm. Cache only, seconds.
#
# Both detectors read the same simulations, so a switch would cost a
# documentation update, not a re-run. The last section is why the answer is
# not simply yes.
# ============================================================================
if "STAT_SW" not in globals():
    raise SystemExit("Run the first two cells first.")

import json
import numpy as np
import paths

BUDGET_HERE = 1
CAND = {
    "one-sided t": (ps.detect_cp,    "thresh", 6.0),
    "two-sided t": (ps.detect_cp2,   "thresh", None),
    "CUSUM":       (da.detect_cusum, "h",      None),
}

# calibrate the two that the earlier cells calibrate; keep the one-sided
# detector at the threshold DETECTION_RESULTS.md actually used
for nm, key in (("two-sided t", "Max-split t (ours)"), ("CUSUM", "CUSUM")):
    fn, param, _ = CAND[nm]
    CAND[nm] = (fn, param, float(calib(key, BUDGET_HERE)))
for nm, (fn, param, x) in CAND.items():
    print(f"  {nm:<14} {param} = {x:g}")


def score(cache, nm):
    """Catches, early alarms and mean latency. switch_s comes from the run's
    own settings, because runs_stim_long_a3500 switches at 10 s."""
    fn, param, x = CAND[nm]
    runs = rs.load_cached(cache)
    sw = json.loads(str(np.load(paths.cache(cache) / "seed1.npz")["settings"]))["hold_ms"] / 1000.0
    r = ps.evaluate(rs.pair_average(runs), fn, switch_s=sw, **{param: x})
    return r["ok"], r["n"], len(r["false"]), r["latency"]


ROWS = [
    ("no stimulation",         "runs_stim_a0"),
    ("130 Hz / 250 uA",        "runs_stim_a250"),
    ("180 Hz / 500 uA",        "runs_stim_a500"),
    ("130 Hz / 1000 uA",       "runs_stim_a1000"),
    ("130 Hz / 3500 uA",       "runs_stim_a3500"),
    ("130 Hz / 3500 uA, 10 s", "runs_stim_long_a3500"),
    ("up,   0.3 -> 0.7",       "runs_dir_0.3to0.7_stim180_500"),
    ("down, 0.7 -> 0.3",       "runs_dir_0.7to0.3_stim180_500"),
]

print("\n" + "=" * 82)
print("DETECTION_RESULTS.md ROWS, THREE DETECTORS")
print("=" * 82)
print(f"  {'row':<24}" + "".join(f"{k:>19}" for k in CAND))
print(f"  {'':<24}" + "".join(f"{'caught  lat  early':>19}" for _ in CAND))
for label, cache in ROWS:
    cells = []
    for nm in CAND:
        ok, n, early, lat = score(cache, nm)
        lt = f"{lat:5.2f}" if ok else "   --"
        cells.append(f"{ok:>2}/{n:<2} {lt}  {early:>2}e")
    print(f"  {label:<24}" + "".join(f"{c:>19}" for c in cells))
print("\n  'e' counts alarms BEFORE the switch. Those are false alarms.")

print("\n" + "=" * 82)
print("THE LONG-BASELINE RUN IS THE ONE THAT MATTERS")
print("=" * 82)
print("""  Every row above has 5 s of baseline before the switch except the last,
  which has 10 s. On that one row every detector that can fire downward
  alarms on all 10 runs before anything happens:
""")
p_long = rs.pair_average(rs.load_cached("runs_stim_long_a3500"))
for nm, fn, kw in [
        ("one-sided t",  ps.detect_cp,          dict(thresh=6.0, min_size=3)),
        ("two-sided t",  ps.detect_cp2,         dict(thresh=CAND["two-sided t"][2], min_size=3)),
        ("CUSUM",        da.detect_cusum,       dict(h=CAND["CUSUM"][2])),
        ("Page-Hinkley", da.detect_page_hinkley, dict(h=14.2)),
        ("EWMA",         da.detect_ewma,        dict(L=4.2)),
        ("Shewhart",     da.detect_shewhart,    dict(L=3.9))]:
    r = ps.evaluate(p_long, fn, switch_s=10.0, **kw)
    lt = f"{r['latency']:.2f}s" if r["ok"] else "   --"
    print(f"    {nm:<14} caught {r['ok']:>2}/10   false alarms {len(r['false']):>2}/10   {lt}")

print("""
  Only the one-sided test survives, and it survives by being unable to fire
  downward at all. Every null run used to set these thresholds has a 5 s
  baseline, so none of them could have exposed this.

  Two explanations fit, and the cached data cannot separate them:
    - baseline length. A longer stretch gives any two-sided method more
      chances to find a downward fluctuation worth alarming on.
    - amplitude. This row is also the 3500 uA condition, where the
      period-2 measurement artifact lives.

  The run that separates them is a long-baseline set at 500 uA, which is not
  cached. That is about 20 minutes of MATLAB and it decides whether the
  detector can be changed at all.""")

print("\n" + "=" * 82)
print("""READ THIS BEFORE SWITCHING

  On 5 s baselines CUSUM beats the current detector on every row, catching
  the same or more and alarming roughly half a second sooner.

  On the only 10 s baseline available it fails completely, as does every
  other two-sided method including the two-sided version of our own test
  that section 5 of DETECTION_RESULTS.md recommends.

  A deployed detector sees long baselines, not 5 s ones, so the long row is
  the realistic case and the short rows are the convenient one. Settle the
  500 uA long-baseline run before changing anything.""")
